# Batch Prompt Experiments (All Students)

This notebook runs prompt-strategy experiments on a student cohort (batch mode),
using the same output style as the single-student experiments.

Experiments:
1) Single submission in batch context
2) Small sample (5 submissions)
3) Full cohort run (one representative submission per student)
4) Strategy summary

In [ ]:
# 1) Configuration and Shared Utilities
import pandas as pd

from lib.experiment_utils import (
    create_client,
    load_best_attempts_df,
    select_cohort_ids,
    get_representative_rows_per_student,
    build_strategies,
    run_experiment_rows,
    build_strategy_summary,
    save_results,
)

MODEL_ID = "gemini-2.5-flash"
RANDOM_SEED = 42
BATCH_SIZE = 50
MIN_SUBMISSIONS = 5

client = create_client()
print(f"Environment ready. Using model: {MODEL_ID}")

In [ ]:
# 2) Load Data and Define Batch Cohort
best_attempts_df = load_best_attempts_df()
cohort_ids = select_cohort_ids(
    best_attempts_df=best_attempts_df,
    batch_size=BATCH_SIZE,
    min_submissions=MIN_SUBMISSIONS,
    seed=RANDOM_SEED,
)
if not cohort_ids:
    raise ValueError("No eligible students found for batch experiments.")

cohort_df = best_attempts_df[best_attempts_df["SubjectID"].isin(cohort_ids)].copy()
print(f"Batch cohort size: {len(cohort_ids)}")
display(cohort_df.head())

In [ ]:
# 3) Strategies and Experiment Runner
strategies = build_strategies(
    focus_problem_ids=list(cohort_df["ProblemID"].unique())
)


def run_rows(rows_df: pd.DataFrame, sleep_seconds: float = 1.0) -> pd.DataFrame:
    return run_experiment_rows(
        rows_df=rows_df,
        client=client,
        model_id=MODEL_ID,
        strategies=strategies,
        sleep_seconds=sleep_seconds,
    )


print("Batch experiment runner configured.")

## Experiment 1: Single Submission (Batch Context)

In [ ]:
single_target = cohort_df.sort_values(["Score", "ProblemID"]).iloc[0]
print(f"Analyzing student {single_target['SubjectID']} / problem {single_target['ProblemID']}")

exp1_df = run_rows(pd.DataFrame([single_target]), sleep_seconds=0)
display_cols = ["SubjectID", "ProblemID", "Score"] + [c for c in exp1_df.columns if c.endswith("_Output")]
display(exp1_df[display_cols])

## Experiment 2: Batch Sample (5 Submissions)

In [ ]:
sample_n = min(5, len(cohort_df))
exp2_sample = cohort_df.sample(n=sample_n, random_state=RANDOM_SEED)

exp2_df = run_rows(exp2_sample, sleep_seconds=1)
display(exp2_df[["SubjectID", "ProblemID", "Score"] + [c for c in exp2_df.columns if c.endswith("_Output")]])

## Experiment 3: Full Batch Run (One Submission per Student)
For each student in the cohort, we analyze one representative submission (lowest score).

In [ ]:
rep_df = get_representative_rows_per_student(cohort_df, cohort_ids)
print(f"Running full batch analysis on {len(rep_df)} representative submissions...")

exp3_df = run_rows(rep_df, sleep_seconds=1)
display(exp3_df)

exp3_csv = f"batch_{len(rep_df)}_prompt_results.csv"
save_results(exp3_df, exp3_csv)
print(f"Saved: {exp3_csv}")

## Experiment 4: Strategy Summary

In [ ]:
if 'exp3_df' not in locals() or exp3_df.empty:
    raise ValueError("Run Experiment 3 first.")

exp4_summary_df = build_strategy_summary(exp3_df, strategies)
display(exp4_summary_df)

exp4_csv = f"batch_{len(exp3_df)}_strategy_summary.csv"
save_results(exp4_summary_df, exp4_csv)
print(f"Saved: {exp4_csv}")